# Qwen2.5-Coder-3B — Commit Message Evaluation

This notebook evaluates the LoRA-adapted Qwen2.5-Coder-3B model in the `OffHand` project. It performs an early CUDA check, clears unused PyTorch GPU cache before loading the model, locates project files, loads the test dataset, generates predictions, and computes available text-generation metrics.

**Important:** CUDA must be available before continuing. The notebook deliberately stops with an actionable diagnostic if the active Jupyter kernel has CPU-only PyTorch.

## 1. Configure CUDA allocator and clean up memory

This follows the setup pattern from the working DeepSeek QLoRA notebook: set the allocator environment variable before importing PyTorch, run garbage collection, and clear the CUDA cache before model loading.

In [1]:
import os

# Must be set before importing torch.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

'false'

In [2]:
import gc
import sys
import subprocess
import platform
from pathlib import Path

import torch

gc.collect()

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("Allocated memory (GB):", round(torch.cuda.memory_allocated(0) / 1024**3, 3))
    print("Reserved memory (GB):", round(torch.cuda.memory_reserved(0) / 1024**3, 3))
else:
    print("\nCUDA is NOT available in this Jupyter kernel.")
    print("Install a CUDA-enabled PyTorch build into THIS exact interpreter:")
    print(f'"{sys.executable}" -m pip uninstall -y torch torchvision torchaudio')
    print(f'"{sys.executable}" -m pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121')
    print("\nThen restart the Jupyter kernel and rerun this cell.")
    raise RuntimeError(
        "Stopping before model loading: this kernel has CPU-only/unavailable CUDA. "
        "Use the install command printed above with this kernel's Python executable."
    )

Python executable: c:\Users\ASUSD\AppData\Local\Programs\Python\Python311\python.exe
Python version: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
PyTorch version: 2.5.1+cu121
PyTorch CUDA build: 12.1
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
GPU count: 1
Allocated memory (GB): 0.0
Reserved memory (GB): 0.0


## 2. Install / verify notebook dependencies

Run this only if an import in a later cell is missing. It installs into the current notebook kernel, not an unrelated system Python.

In [3]:
# Uncomment only if needed, then restart the kernel after installation.
# %pip install -U transformers peft accelerate bitsandbytes datasets pandas openpyxl evaluate sacrebleu rouge-score nltk

## 3. Locate the project and configure model/data paths

The project root is detected by finding `commit-model-lora/adapter_config.json` while walking upward from the notebook directory and common project locations. If auto-detection fails, set `PROJECT_ROOT` manually.

In [4]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()

def find_project_root(start: Path):
    candidates = [start, *start.parents]
    # Also check the known project location from this project's setup.
    candidates.append(Path.home() / "Desktop" / "Programing" / "seniorProject" / "OffHand")
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "commit-model-lora" / "adapter_config.json").is_file():
            return candidate
    return None

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

# If auto-detection returns None, uncomment and edit this line:
# PROJECT_ROOT = Path(r"C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root (expected commit-model-lora/adapter_config.json). "
        "Set PROJECT_ROOT manually in this cell."
    )

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
ADAPTER_DIR = PROJECT_ROOT / "commit-model-lora"

print("Notebook working directory:", NOTEBOOK_DIR.resolve())
print("Project root:", PROJECT_ROOT)
print("Adapter directory:", ADAPTER_DIR)

Notebook working directory: C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand\notebooks
Project root: C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand
Adapter directory: C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand\commit-model-lora


## 4. Find and validate the LoRA adapter

A LoRA adapter requires both `adapter_config.json` and its trained weight file. The configuration file alone is not sufficient to load the fine-tuned model.

In [5]:
# -------------------- Adapter Validation --------------------

adapter_config = ADAPTER_DIR / "adapter_config.json"

adapter_weight_candidates = [
    ADAPTER_DIR / "adapter_model.safetensors",
    ADAPTER_DIR / "adapter_model.bin",
]

adapter_weights = next(
    (p for p in adapter_weight_candidates if p.is_file()),
    None
)

if not adapter_config.is_file():
    print(f"WARNING: Adapter config not found: {adapter_config}")
    USE_LORA = False

elif adapter_weights is None:
    files_here = (
        [p.name for p in ADAPTER_DIR.iterdir()]
        if ADAPTER_DIR.exists() else []
    )

    print("WARNING: Trained LoRA weights are missing.")
    print("Adapter directory:", ADAPTER_DIR)
    print("Files found:", files_here)
    print("Continuing with the original Qwen base model.")
    USE_LORA = False

else:
    print("LoRA adapter found.")
    USE_LORA = True

print("Adapter config:", adapter_config)
print("Adapter weights:", adapter_weights)
print("Use LoRA:", USE_LORA)

Adapter directory: C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand\commit-model-lora
Files found: ['adapter_config.json', 'chat_template.jinja', 'README.md', 'tokenizer.json', 'tokenizer_config.json']
Continuing with the original Qwen base model.
Adapter config: C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand\commit-model-lora\adapter_config.json
Adapter weights: None
Use LoRA: False


## 5. Configure test data and model

The default base model is `Qwen/Qwen2.5-Coder-3B-Instruct`. Change `BASE_MODEL_NAME` only if the adapter was trained from a different exact base model. The test-file locator searches common project data folders for CSV, Excel, JSON, JSONL, and Parquet files; explicitly set `TEST_FILE` if several candidates exist.

In [6]:
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

# Set this to the exact test dataset file if automatic discovery chooses the wrong file.
TEST_FILE = None  # Example: PROJECT_ROOT / "data" / "commitbench_test.csv"

DATA_EXTENSIONS = {".csv", ".xlsx", ".xls", ".json", ".jsonl", ".parquet"}
EXCLUDED_DIRS = {
    ".git", ".venv", "venv", "__pycache__", "node_modules",
    ".cache", "commit-model-lora", "outputs", "checkpoints"
}

def discover_test_files(root: Path):
    found = []
    preferred_dirs = [
        root / "data", root / "dataset", root / "datasets",
        root / "test", root / "evaluation", root
    ]
    visited = set()
    for base in preferred_dirs:
        if not base.exists():
            continue
        for path in base.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in DATA_EXTENSIONS:
                continue
            if any(part.lower() in EXCLUDED_DIRS for part in path.parts):
                continue
            if path.resolve() in visited:
                continue
            visited.add(path.resolve())
            name = path.name.lower()
            score = 0
            if "test" in name: score += 5
            if "commitbench" in name: score += 4
            if "commit" in name: score += 2
            if "eval" in name: score += 1
            found.append((score, path))
    return [p for _, p in sorted(found, key=lambda x: (-x[0], len(str(x[1]))))]

if TEST_FILE is None:
    candidates = discover_test_files(PROJECT_ROOT)
    print("Possible test files:")
    for p in candidates[:20]:
        print(" -", p.relative_to(PROJECT_ROOT))
    if not candidates:
        raise FileNotFoundError(
            "No supported dataset file was found. Set TEST_FILE to your test CSV/XLSX/JSON/JSONL/Parquet file."
        )
    TEST_FILE = candidates[0]

TEST_FILE = Path(TEST_FILE).resolve()
if not TEST_FILE.is_file():
    raise FileNotFoundError(f"Test dataset not found: {TEST_FILE}")

print("\nSelected test file:", TEST_FILE)
print("Base model:", BASE_MODEL_NAME)

Possible test files:
 - data\raw\test.csv
 - data\multitask\test.jsonl
 - data\processed\test.jsonl
 - CommitSuite\Ten-category-eval_dataset\test.json
 - CommitSuite\all_data\gotd_td\commits.json
 - CommitSuite\all_data\a-h_templ\commits.json
 - CommitSuite\all_data\ddev_ddev\commits.json
 - CommitSuite\all_data\ory_hydra\commits.json
 - CommitSuite\all_data\rmwc_rmwc\commits.json
 - CommitSuite\all_data\aws_aws-cdk\commits.json
 - CommitSuite\all_data\cosmos_gaia\commits.json
 - CommitSuite\all_data\google_uuid\commits.json
 - CommitSuite\all_data\kumahq_kuma\commits.json
 - CommitSuite\all_data\lerna_lerna\commits.json
 - CommitSuite\all_data\oclif_oclif\commits.json
 - CommitSuite\all_data\wandb_wandb\commits.json
 - CommitSuite\all_data\caarlos0_env\commits.json
 - CommitSuite\all_data\caarlos0_svu\commits.json
 - CommitSuite\all_data\go-task_task\commits.json
 - CommitSuite\all_data\oddlama_vane\commits.json

Selected test file: C:\Users\ASUSD\Desktop\Programing\seniorProject\OffH

## 6. Load and inspect the test dataset

The notebook attempts to identify the diff/input and reference commit-message columns from common names. If automatic detection is wrong, set `INPUT_COLUMN` and `REFERENCE_COLUMN` manually after inspecting `df.columns`.

In [7]:
import pandas as pd
import json

suffix = TEST_FILE.suffix.lower()
if suffix == ".csv":
    df = pd.read_csv(TEST_FILE)
elif suffix in {".xlsx", ".xls"}:
    df = pd.read_excel(TEST_FILE)
elif suffix == ".json":
    df = pd.read_json(TEST_FILE)
elif suffix == ".jsonl":
    df = pd.read_json(TEST_FILE, lines=True)
elif suffix == ".parquet":
    df = pd.read_parquet(TEST_FILE)
else:
    raise ValueError(f"Unsupported test file type: {suffix}")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))

Dataset shape: (249688, 6)
Columns: ['hash', 'diff', 'message', 'project', 'split', 'diff_languages']


,hash,diff,message,project,split,diff_languages
0,713a0a528e7b8ab7348d8e43d3e3aa2e234b20ab,diff --git a/Auth/OpenID/Message.php b/Auth/Op...,[project @ Fix openid.ns value],openid_php-openid,test,php
1,626e13a211c82f833ade88faabcfa104bbf6c7f6,diff --git a/salt/modules/systemd.py b/salt/mo...,fix bug from systems without any legacy init s...,saltstack_salt,test,py
2,b0bcb691ff789314ec525a74ec0a505e57a20a79,diff --git a/pyvex/lifting/util/vex_helper.py ...,feature: add support for `Iop_Sar<width>` arit...,angr_pyvex,test,py


In [8]:
def choose_column(columns, preferred_names):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in preferred_names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    # fallback: substring matching
    for name in preferred_names:
        for normalized, original in lookup.items():
            if name.lower() in normalized:
                return original
    return None

INPUT_COLUMN = choose_column(df.columns, [
    "diff", "patch", "code_diff", "commit_diff", "input", "prompt", "source"
])
REFERENCE_COLUMN = choose_column(df.columns, [
    "message", "commit_message", "target", "reference", "ground_truth", "completion"
])

print("Detected input column:", INPUT_COLUMN)
print("Detected reference column:", REFERENCE_COLUMN)

# If needed, set these explicitly, for example:
# INPUT_COLUMN = "diff"
# REFERENCE_COLUMN = "message"

if INPUT_COLUMN is None or REFERENCE_COLUMN is None:
    raise ValueError(
        "Could not identify input/reference columns. Inspect df.columns and set "
        "INPUT_COLUMN and REFERENCE_COLUMN manually in this cell."
    )

df = df.dropna(subset=[INPUT_COLUMN, REFERENCE_COLUMN]).copy()
df[INPUT_COLUMN] = df[INPUT_COLUMN].astype(str)
df[REFERENCE_COLUMN] = df[REFERENCE_COLUMN].astype(str)
df = df[(df[INPUT_COLUMN].str.strip() != "") & (df[REFERENCE_COLUMN].str.strip() != "")]
df = df.reset_index(drop=True)

print("Usable evaluation rows:", len(df))

Detected input column: diff
Detected reference column: message
Usable evaluation rows: 249688


## 7. Load tokenizer, base model, and LoRA adapter

This uses 4-bit NF4 quantization to reduce VRAM usage on the RTX 3070 Ti Laptop GPU. If `bitsandbytes` is not supported by the installed Windows environment, use a compatible Linux/WSL2 environment or load the base model in FP16 if sufficient VRAM is available.

In [9]:

import torch
import importlib.util
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# --------------------------------------------------
# 1. Check CUDA
# --------------------------------------------------

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Check your PyTorch installation.")

print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# 2. Check adapter files BEFORE importing PEFT
# --------------------------------------------------

ADAPTER_DIR = Path(ADAPTER_DIR)

adapter_weights = [
    ADAPTER_DIR / "adapter_model.safetensors",
    ADAPTER_DIR / "adapter_model.bin"
]

has_lora_weights = (
    (ADAPTER_DIR / "adapter_config.json").is_file()
    and any(p.is_file() for p in adapter_weights)
)

print("Adapter exists:", has_lora_weights)

# --------------------------------------------------
# 3. Load tokenizer
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

# --------------------------------------------------
# 4. Configure model loading
# --------------------------------------------------

bnb_available = importlib.util.find_spec("bitsandbytes") is not None

if bnb_available:
    print("BitsAndBytes detected. Using 4-bit quantization.")

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

else:
    print("BitsAndBytes unavailable. Using FP16 instead.")

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

# --------------------------------------------------
# 5. Load LoRA adapter if available
# --------------------------------------------------

if has_lora_weights:

    print("Loading PEFT adapter...")

    try:
        from peft import PeftModel

        model = PeftModel.from_pretrained(
            base_model,
            str(ADAPTER_DIR),
            is_trainable=False
        )

        print("Trained LoRA adapter loaded successfully.")

    except Exception as e:
        print("PEFT loading failed:")
        print(type(e).__name__, str(e))
        raise

else:
    model = base_model

    print("WARNING: LoRA weights are missing.")
    print("Using the original Qwen base model.")

# --------------------------------------------------
# 6. Finalize
# --------------------------------------------------

model.eval()

print("\nModel loaded successfully.")
print(
    "Input embedding device:",
    model.get_input_embeddings().weight.device
)


c:\Users\ASUSD\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.5.1+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
Adapter exists: False
BitsAndBytes unavailable. Using FP16 instead.


ImportError: 
AutoModelForCausalLM requires PyTorch >= 2.6, but PyTorch was not found or an older version is installed.
Please install or upgrade PyTorch by following the instructions on the installation page:
https://pytorch.org/get-started/locally/
Please note that you may need to restart your runtime after installation.


## 8. Build prompts and generate predictions

The prompt template below is for code-diff-to-commit-message generation. If your fine-tuning notebook used a different prompt template, copy that exact template here to keep evaluation consistent.

In [ ]:
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 64
BATCH_SIZE = 1  # conservative for a 3B model on a 3070 Ti Laptop GPU

def build_prompt(diff_text):
    return (
        "You are an expert software developer. Write a concise, informative "
        "Git commit message that summarizes the following code changes. "
        "Return only the commit message.\n\n"
        f"Code diff:\n{diff_text}\n\n"
        "Commit message:"
    )

@torch.inference_mode()
def generate_commit_message(diff_text):
    prompt = build_prompt(diff_text)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    generated_tokens = output[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return text.strip().splitlines()[0].strip() if text.strip() else ""

predictions = []
for start in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[start:start + BATCH_SIZE]
    for value in batch[INPUT_COLUMN].tolist():
        predictions.append(generate_commit_message(value))
    if (start // BATCH_SIZE + 1) % 25 == 0:
        print(f"Generated {len(predictions)}/{len(df)}")
        gc.collect()
        torch.cuda.empty_cache()

df["prediction"] = predictions
df[["prediction", REFERENCE_COLUMN]].head(10)

## 9. Compute automatic evaluation metrics

BLEU, ROUGE-L, and METEOR are calculated when their packages are installed. These lexical metrics do not fully measure semantic correctness, so interpret them alongside manual review.

In [ ]:
# If imports fail, run:
# %pip install -U evaluate sacrebleu rouge-score nltk

import numpy as np

references = df[REFERENCE_COLUMN].astype(str).tolist()
preds = df["prediction"].astype(str).tolist()

metric_results = {}

try:
    import evaluate
    bleu_metric = evaluate.load("sacrebleu")
    metric_results["BLEU"] = bleu_metric.compute(
        predictions=preds,
        references=[[r] for r in references]
    )["score"]
except Exception as e:
    metric_results["BLEU"] = f"Unavailable: {e}"

try:
    import evaluate
    rouge_metric = evaluate.load("rouge")
    metric_results["ROUGE"] = rouge_metric.compute(
        predictions=preds,
        references=references
    )
except Exception as e:
    metric_results["ROUGE"] = f"Unavailable: {e}"

try:
    import evaluate
    meteor_metric = evaluate.load("meteor")
    metric_results["METEOR"] = meteor_metric.compute(
        predictions=preds,
        references=references
    )["meteor"]
except Exception as e:
    metric_results["METEOR"] = f"Unavailable: {e}"

metric_results

## 10. Save predictions and metrics

Results are written under `PROJECT_ROOT/evaluation_results`. The source dataset is not modified.

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

predictions_path = RESULTS_DIR / "qwen25_coder_3b_predictions.csv"
metrics_path = RESULTS_DIR / "qwen25_coder_3b_metrics.json"

df.to_csv(predictions_path, index=False, encoding="utf-8-sig")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metric_results, f, indent=2, ensure_ascii=False, default=str)

print("Predictions saved to:", predictions_path)
print("Metrics saved to:", metrics_path)

## 11. Inspect sample outputs

In [ ]:
for idx in range(min(10, len(df))):
    print("=" * 90)
    print("REFERENCE:", df.loc[idx, REFERENCE_COLUMN])
    print("PREDICTION:", df.loc[idx, "prediction"])

## Troubleshooting

- If `torch.version.cuda` is `None`, the active notebook kernel is importing CPU-only PyTorch. Use the exact `sys.executable` printed in Section 1 to install the CUDA wheel, then restart the kernel.
- If CUDA is available but `bitsandbytes` fails to import or quantize on Windows, the 4-bit loading backend may not be supported in that environment. Use WSL2/Linux with a compatible CUDA and bitsandbytes installation, or switch to FP16 loading if VRAM permits.
- If the adapter weights are missing, restore `adapter_model.safetensors` (preferred) or `adapter_model.bin` into `commit-model-lora`. `adapter_config.json` alone is not the trained adapter.
- If the test dataset auto-detection selects the wrong file, set `TEST_FILE` explicitly and verify `INPUT_COLUMN` and `REFERENCE_COLUMN`.